In [73]:
import netket as nk
import jax.numpy as jnp
import numpy as np

### Potential

In [74]:
import jax
import jax.numpy as jnp
import numpy as np

from functools import partial


@partial(jax.jit, static_argnums=(1,))
def smallest_vecs(basis, n):
    dim = basis.shape[-1]
    r = np.arange(-n, (n + 1))  # produces O(n*n*n) numbers

    vecs = jnp.array(np.meshgrid(*(dim * (r,)))).T.reshape(-1, dim)
    return vecs


def ewald_coulomb(x, L, alpha, sdim=3, kmax=20, cutoff=None):
    """Ewald summation for cubic lattices."""

    if sdim != 3:
        raise ValueError("Ewald summation only implemented for 3 spatial dimensions.")

    if cutoff is None:
        cutoff = 2.0 * L

    N = x.shape[0] // sdim
    x = x.reshape(-1, N, sdim)

    dists = x[..., :, None, :] - x[..., None, :, :]
    V = L**sdim

    def real_sum(dists):
        """Real space part of Ewald summation"""

        # vec(R)=0
        mask = 1 - jnp.eye(N)
        d = jnp.linalg.norm(dists, axis=-1)
        res1 = jnp.where(
            mask[jnp.newaxis, :], jax.scipy.special.erfc(alpha * d) / d, 0
        ).reshape(-1)

        # vec(R)=L * vec(n)
        Rn = smallest_vecs(jnp.eye(sdim), 2) * L
        temp = jnp.linalg.norm(Rn, axis=-1)
        Rn = jnp.where(temp[:, None] <= cutoff, Rn, 0)  # exlude too far away neighbours
        temp = jnp.linalg.norm(Rn, axis=-1)
        d = jnp.linalg.norm(dists[..., None, :] + Rn, axis=-1)

        res2 = jnp.where(temp == 0, 0, jax.scipy.special.erfc(alpha * d) / d).reshape(
            -1
        )  # Rn=0 is special

        res = jnp.sum(res1) + jnp.sum(res2)
        return res

    def recip_sum(x):
        """Reciprocal space part of Ewald summation"""

        # vec(k)=1/L * vec(n)
        Gm = smallest_vecs(jnp.eye(sdim), kmax) / L
        # |vec(k)|>kmax
        G = jnp.linalg.norm(Gm, axis=-1, keepdims=True)
        Gm = jnp.where(G <= kmax / L, Gm, 0)
        G = jnp.linalg.norm(Gm, axis=-1)
        G2 = (G**2).reshape(-1)

        t1 = jnp.where(G2 == 0, 0, jnp.exp(-G2 * jnp.pi**2 / (alpha**2)) / G2)
        # structure factor
        S = jnp.sum(
            jnp.where(
                G2 == 0,
                0,
                jnp.exp(1j * 2 * jnp.pi * jnp.einsum("...ij,kj->...ik", x, Gm)),
            ),
            axis=-2,
        ).reshape(-1)

        t2 = jnp.abs(S) ** 2

        res = 1 / (V * jnp.pi) * jnp.sum(t1 * t2)
        return res

    def self_energy():
        return -N * alpha / jnp.sqrt(jnp.pi)

    def constant():
        V = L**sdim
        return -(N**2) * jnp.pi / (2 * alpha**2 * V)

    ereal = real_sum(dists)
    erec = recip_sum(x)
    eself = self_energy()
    const = constant()

    return 0.5 * ereal + 0.5 * erec + eself + const


## MLP

In [75]:
# Example conversion

from flax import linen as nn
import jax
import jax.numpy as jnp


class MLP(nn.Module):
    hidden_dim: int
    output_dim: int

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(self.hidden_dim)(x)
        x = nn.relu(x)
        x = nn.Dense(self.output_dim)(x)
        return x


# model = MLP(hidden_dim=64, output_dim=10)

# key = jax.random.key(0)
# x = jnp.ones((32, 20))

# variables = model.init(key, x)
# y = model.apply(variables, x)


# using NNX:

from flax import nnx
import jax.numpy as jnp
class MLP_NNX(nnx.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, *, rngs):
        self.linear1 = nnx.Linear(
            input_dim, hidden_dim, rngs=rngs
        )
        self.linear2 = nnx.Linear(
            hidden_dim, output_dim, rngs=rngs
        )

    def __call__(self, x):
        x = self.linear1(x)
        x = nnx.relu(x)
        x = self.linear2(x)
        return x
model_nnx = MLP_NNX(input_dim=20, hidden_dim=64, output_dim=10, rngs=nnx.Rngs(0))
x = jnp.ones((32, 20))
y = model_nnx(x)
print(y.shape)

(32, 10)


In [76]:
from flax import nnx
from typing import Any, Callable, Tuple
import jax.numpy as jnp

class MLP(nnx.Module):
    def __init__(
        self,
        in_dim: int,
        out_dim: int,
        hidden_layers: tuple[int, ...],
        *,
        rngs: nnx.Rngs,
        activation: Callable = nnx.gelu,
        output_activation: Callable | None = None,
        last_bias: bool = True,
        last_linear: bool = True,
        auto_width: bool = False,
        dtype: Any = jnp.float32,
        kwargs: dict | None = None,
    ):
        self.activation = activation
        self.output_activation = output_activation
        self.last_linear = last_linear

        kwargs = kwargs or {}

        if auto_width and len(hidden_layers) == 1:
            n_hidden = hidden_layers[0]
            qs = [ k / ( n_hidden+1 ) for k in range(n_hidden + 2) ]
            dims = [ int( jnp.round(in_dim**(1-q) * out_dim**q )) for q in qs ]
        else:
            dims = [in_dim, *hidden_layers, out_dim]

        self.layers = nnx.List()
        for k in range(len(dims)-1):
            last = k == len(dims) - 2
            use_bias = (not last or last_bias or not last_linear)
            self.layers.append(
                nnx.Linear(
                    in_features=dims[k],
                    out_features=dims[k+1],
                    use_bias=use_bias,
                    param_dtype=dtype,
                    rngs=rngs,
                    **kwargs
                )
            )

    def __call__(self, x):
        for k, layer in enumerate(self.layers):
            last = k == len(self.layers) - 1

            x = layer(x)
            if last:
                if self.output_activation is not None:
                    x = self.output_activation(x)
                if not self.last_linear:
                    x = self.activation(x)
            else:
                x = self.activation(x)
        return x

model = MLP(in_dim=4, 
            out_dim=2,
            hidden_layers=(64,64),
            rngs=nnx.Rngs(0)    
        )

x = jnp.ones((32,4))
y = model(x)
print(y.shape)

(32, 2)


### Slater Determinant

In [85]:
from flax import nnx
from typing import Any, Callable, Sequence, Tuple
import scipy

class LogSlaterDet(nnx.Module):
    def __init__(
            self,
            n_per_spin: Tuple[int],
            orbitals: Sequence[Callable[..., Any]],
            return_signs: bool = False,
            add_signs: bool = True,
    ):
        self.n_per_spin = n_per_spin
        self.orbitals = nnx.data(orbitals)
        self.return_signs = return_signs
        self.add_signs = add_signs

    def __call__(self, x, orbs=None, mask=True):
        if not self.orbitals:
            raise ValueError("Empty LogslaterDet module")

        assert(x.ndim == 3)
        N = x.shape[-2]

        mat = []
        for key, value in self.orbitals.items():
            o = value(x)
            mat.append(o)
        mat = jnp.concatenate(mat, axis=-1)

        if mask:
            x = jnp.tile(
                jnp.array(self.n_per_spin[0]*[1] + self.n_per_spin[1]*[0])[:, None],
                (1, self.n_per_spin[0])
            )
            y = jnp.tile(
                jnp.array(self.n_per_spin[0]*[0] + self.n_per_spin[1]*[1])[:, None],
                (1, self.n_per_spin[1])
            )
            mask = jnp.concatenate((x, y), axis=-1)
            mat = mask[...,:,:] * jnp.exp(mat)

        if orbs is not None:
            mat = mat * jnp.exp(orbs)

        signs, logslaterdet = jnp.linalg.slogdet(mat)

        # normalization
        log_norm = 0.5 * jnp.log(scipy.special.factorial(N))
        logslaterdet -= log_norm

        if self.add_signs:
            logslaterdet = logslaterdet + jnp.log(signs)

        if self.return_signs:
            return signs, logslaterdet
        else:
            return logslaterdet


### Plane-Wave

In [86]:
from flax import nnx
from netket.utils import HashableArray
from netket.utils.types import DType, NNInitFunc
from jax.nn.initializers import lecun_normal

class PW(nnx.Module):
    def __init__(
            self,
            kvecs: HashableArray,
            n_orbs: int,
            *,
            rngs: nnx.Rngs,
            combine: bool = False,
            param_init = nnx.initializers.lecun_normal(),
            dtype = jnp.float32
    ):
        self.kvecs = nnx.data(kvecs)
        self.n_orbs = n_orbs
        self.combine = combine
        self.dtype = dtype

        if combine:
            n_k = kvecs.shape[0]
            self.coefficients = nnx.Param(
                param_init(
                    rngs.params(),
                    (n_k, n_orbs),
                    dtype
                )
            )

    def __call__(self, x):
        res = 1j * jnp.einsum("ij,...j->...i", self.kvecs, x) # ikx #(...,N, N_k)
        if self.combine:
            res = jnp.einsum("ij,...i->...j", self.coefficients, res)

        # res = jnp.einsum("...nk,ko->...no",res,self.coefficients)
        return res
    
kvecs_example = jnp.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [0.0, -1.0],
]) # Shape: (Nk, D)

plane_wave = PW(kvecs=kvecs_example, n_orbs=4, rngs=nnx.Rngs(0))

example_position_x = jnp.array([
    [0.1, 0.2],
    [0.5, 0.3],
    [0.8, 0.7],
]) # shape: (N, D)
res = plane_wave(example_position_x)
print(res)
print(res.shape)


[[ 0.+0.1j  0.+0.2j  0.+0.1j -0.-0.2j]
 [ 0.+0.5j  0.+0.3j  0.+0.5j -0.-0.3j]
 [ 0.+0.8j  0.+0.7j  0.+0.8j -0.-0.7j]]
(3, 4)


Shape: 
K: (4,2)
x: (3,2)

res = jnp.einsum("ij,...j->...i", kvecs, x)
r_nk = i \sum_d k_kd x_nd = i k_k x_n

output: (3,4)

-> 3 particles × 4 plane waves

In [87]:
class Networks:
    def __init__(
        self,
        intermediate_dim: int, 
        num_intermediate: int, 
        embedding_dim: int,
        *,
        rngs: nnx.Rngs,
        activation: Callable = nnx.gelu
    ):
        """Create networks for the MPNQS
        Args:
            intermediate_dim: dimension of the intermediate coordinates
            num_intermediate: number of hidden layers
            embedding_dim: dimension of the final coordinates of the layer
        """
        self.intermediate_dim = intermediate_dim
        self.num_intermediate = num_intermediate
        self.embedding_dim = embedding_dim
        self.activation = activation
        self.rngs = rngs

    def Net(self, in_dim: int):
        return MLP(
            in_dim=in_dim,
            out_dim=self.embedding_dim,
            hidden_layers=self.num_intermediate * (self.intermediate_dim,),
            activation=self.activation,
            rngs=self.rngs
        )

network_1 = Networks
print(network_1.Net)

<function Networks.Net at 0x11831b100>


In [88]:
from flax import nnx
from typing import Callable
import jax.numpy as jnp


class GraphUpdate(nnx.Module):
    def __init__(
            self,
            networks: Callable,
            L:float,
            n_up: int,
            n_down: int,
            attention_dim: int,
            edge_dim: int,
            *,
            rngs: nnx.Rngs
    ):
        self.L = L
        self.n_up = n_up
        self.n_down = n_down
        self.attention_dim = attention_dim
        embedding_dim = networks.embedding_dim

        self.phi = networks.Net(in_dim=edge_dim)
        self.f = networks.Net(in_dim=3*embedding_dim)
        self.f2 = networks.Net(in_dim=attention_dim)
        self.g = networks.Net(in_dim=edge_dim+embedding_dim)

        init = nnx.initializers.lecun_normal()
        self.wquery = nnx.Param(
            init(
                rngs.params(),
                (edge_dim, attention_dim),
                jnp.float64
            )
        )
        self.wkey = nnx.Param(
            init(
                rngs.params(),
                (edge_dim, attention_dim),
                jnp.float64
            )
        )

    def __call__(self, x, xij, x0, xij0, final):
        # xij: (..., N, N, d_edge)

        xquery = jnp.dot(xij, self.wquery[...])
        xkey = jnp.dot(xij, self.wkey[...])

        # (..., N, N, d_attention)
        attention_logits = jnp.einsum("...ijk,...jlk->...ilk", xquery, xkey) / jnp.sqrt(self.attention_dim)

        weight = self.f2(attention_logits)
        
        message = self.phi(xij) * weight  # ~ edge info x learned particle attention
        xnew = jnp.concatenate(
            (x0, self.f(jnp.concatenate((x, jnp.sum(message, axis=-2)), axis=-1))), axis=-1
        ) # x_i^(n+1) = [ x_i^(0), f(x_i^(n), m_i^(n) ]

        if not final:
            xij = jnp.concatenate(
                (xij0, self.g(jnp.concatenate((xij, message), axis=-1))), axis=-1
            ) # x_ij^(n+1) = [ x_ij^(0), g(x_ij^(n), m_ij^(n) ]
        return xnew, xij

In [89]:
from flax import nnx
print(help(nnx.Embed))

Help on class Embed in module flax.nnx.nn.linear:

class Embed(flax.nnx.module.Module)
 |  Embed(*args: 'Any', **kwargs: 'Any') -> 'Any'
 |
 |  Embedding Module.
 |
 |  Example usage::
 |
 |    >>> from flax import nnx
 |    >>> import jax.numpy as jnp
 |
 |    >>> layer = nnx.Embed(num_embeddings=5, features=3, rngs=nnx.Rngs(0))
 |    >>> nnx.state(layer)
 |    State({
 |      'embedding': Param( # 15 (60 B)
 |        value=Array([[ 0.57966787, -0.523274  , -0.43195742],
 |               [-0.676289  , -0.50300646,  0.33996582],
 |               [ 0.41796115, -0.59212935,  0.95934135],
 |               [-1.0917838 , -0.7441663 ,  0.07713798],
 |               [-0.66570747,  0.13815777,  1.007365  ]], dtype=float32)
 |      )
 |    })
 |    >>> # get the first three and last three embeddings
 |    >>> indices_input = jnp.array([[0, 1, 2], [-1, -2, -3]])
 |    >>> layer(indices_input)
 |    Array([[[ 0.57966787, -0.523274  , -0.43195742],
 |            [-0.676289  , -0.50300646,  0.33996

In [95]:
class MPNN(nnx.Module):
    def __init__(
            self,
            n_up: int,
            n_down: int,
            embedding_dim: int,
            intermediate_dim: int,
            num_intermediate: int,
            attention_dim: int,
            *,
            rngs: nnx.Rngs,
            L: float = 1.0,
            n_interactions: int = 1
    ):
        self.n_up = n_up
        self.n_down = n_down
        self.embedding_dim = embedding_dim
        self.intermediate_dim = intermediate_dim
        self.num_intermediate_dim = num_intermediate
        self.attention_dim = attention_dim
        self.L = L
        self.n_interactions = n_interactions

        self.network_factory = Networks(intermediate_dim, num_intermediate, embedding_dim, rngs=rngs)
        self.X = nnx.Embed(
            num_embeddings=3,
            features=embedding_dim,
            dtype=np.float64,
            param_dtype=np.float64,
            embedding_init=lecun_normal(),
            rngs=rngs
        )
        edge_dim = 2 * sdim + 2 + embedding_dim
        self.layers = nnx.List([GraphUpdate(
                self.network_factory, self.L, self.n_up, self.n_down, self.attention_dim,
                edge_dim, rngs=rngs
            ) for _ in range(n_interactions)])

    def __call__(self, x):
        M, N, sdim = x.shape
        xembed = jnp.tile(self.X(jnp.array([0,1,2]))[None, ...], (M, 1, 1))

        # spin products (assuming first coordinates are spin up and then spin down)
        # s = jnp.hstack((self.n_up * [1], self.n_down * [-1]))
        # ss = jnp.outer(s, s)[..., None]
        s = jnp.concatenate((jnp.ones((self.n_up,)),
                            -jnp.ones((self.n_down,))))
        ss = jnp.outer(s, s)[..., None]

        # compute periodic distances and their norm
        dist = x[..., :, None, :] - x[..., None, :, :]
        d = jnp.linalg.norm(
            jnp.sin(jnp.pi / self.L * dist) + jnp.eye(N)[..., None],
            axis=-1,
            keepdims=True,
        ) * (1.0 - jnp.eye(N)[..., None])
        distp = jnp.concatenate(
            (jnp.sin(2 * jnp.pi / self.L * dist), jnp.cos(2 * jnp.pi / self.L * dist)),
            axis=-1,
        )

        # nodes (translation invariance requires no single particle coordinates)
        x0 = jnp.tile(xembed[:, 0, :][:, None, :], (1, N, 1))
        x = jnp.concatenate(
            (x0, jnp.tile(xembed[:, 1, :][:, None, :], (1, N, 1))), axis=-1
        )

        # edges
        xij0 = jnp.concatenate(
            (distp, d, jnp.tile(ss[None, :, :, :], (x.shape[0], 1, 1, 1))), axis=-1
        )
        xij = jnp.concatenate(
            (xij0, jnp.tile(xembed[:, 2, :][:, None, None, :], (1, N, N, 1))), axis=-1
        )

        for i, layer in enumerate(self.layers):
            final = i == self.n_interactions - 1
            x, xij = layer(x, xij, x0, xij0, final)

        return x

In [96]:
from flax import nnx
from jax.nn.initializers import ones, lecun_normal, variance_scaling

def smallest_kvecs(basis, m, n):
    dim = basis.shape[-1]
    r = np.arange(-n, (n + 1))  # produces O(n*n*n) numbers

    vecs = np.array(np.meshgrid(*(dim * (r,)))).T.reshape(-1, dim)
    vecs = vecs[np.argsort(np.linalg.norm(vecs, axis=1))]
    return vecs[:m, :]


class MPNQS(nnx.Module):
    def __init__(
            self,
            n_per_spin: Tuple[int],
            sdim: int = 1,
            L: float = 1.0,
            paramsre_init = lecun_normal(),
            params_init = variance_scaling(0.0001, "fan_in", "truncated_normal"),
            scale_init = ones,
            dtype = jnp.float64,
            *,
            rngs: nnx.Rngs
    ):
        self.n_per_spin = n_per_spin
        self.sdim = sdim
        self.L = L
        self.dtype = dtype

        self.k_vec = (2*np.pi/L*jnp.concatenate((
            smallest_kvecs(jnp.eye(sdim), self.n_per_spin[0], 10),
            smallest_kvecs(jnp.eye(sdim), self.n_per_spin[1], 10),
        ), axis=0))

        self.orb_basis = nnx.Dict(
            {
                "orbtials_up": PW(self.k_vec[: self.n_per_spin[0]], n_orbs=self.n_per_spin[0], combine=True, rngs=rngs),
                "orbtials_down": PW(self.k_vec[: self.n_per_spin[0]], n_orbs=self.n_per_spin[1], combine=True, rngs=rngs),
            }
        )
        # self.orb_basis["orbtials_up"] = PW(self.k_vec[: self.n_per_spin[0]], n_orbs=self.n_per_spin[0], combine=True, rngs=rngs)
        # self.orb_basis["orbtials_down"] = PW(self.k_vec[self.n_per_spin[0] :], n_orbs=self.n_per_spin[1], combine=True, rngs=rngs)

        embedding_dim = 32
        self.MPNN = MPNN(
            *n_per_spin,
            embedding_dim=embedding_dim,
            intermediate_dim=32,
            num_intermediate=2,
            attention_dim=32,
            L=L,
            n_interactions=1,
            rngs=rngs
        )
        self.slater = LogSlaterDet(n_per_spin=n_per_spin, orbitals=self.orb_basis)
        self.net = MLP(in_dim=2 * embedding_dim + sdim, out_dim=1, hidden_layers=(32,), rngs=rngs)

        y_dim = 2 * embedding_dim

        self.prealup = nnx.Param(paramsre_init(
            rngs.params(), 
            (y_dim, sdim),
            dtype
        ))
        self.pimagup = nnx.Param(params_init(
            rngs.params(),
            (y_dim, self.sdim),
            dtype
        ))

    def __call__(self, x):
        sha = x.shape
        N = sum(self.n_per_spin)
        x = x.reshape(-1, sum(self.n_per_spin), self.sdim)
        M = x.shape[0]

        # backflow
        y = self.MPNN(x)
        pbackflow = (self.prealup + 1j*self.pimagup)
        bf = x + jnp.einsum("kd, ...ik->...id", pbackflow, y)

        k_vec = jnp.tile(self.k_vec[None, None, :, :], (M, N, 1, 1))

        jastrow = jnp.tile(y[:, :, None, :], (1, 1, N, 1))
        jastrow = jnp.concatenate((jastrow, k_vec), axis=-1)
        jastrow = self.net(jastrow).squeeze(axis=-1)
        psi = self.slater(bf, jastrow)

        return psi.reshape(sha[0])


In [ ]:
from jax.sharding import Mesh, AxisType

def mycb(step, logged_data, driver):
    logged_data["acceptance"] = float(driver.state.sampler_state.acceptance)
    return True

N = 14
# unpolarized
n_per_spin = (N // 2, N // 2)
sdim = 3
L = (4 * jnp.pi / 3 * N) ** (1 / 3)
rs = 2.0

geometry = nk.experimental.geometry.Cell(d=3, L=L, pbc=True)
hilb = nk.experimental.hilbert.Particle(N=N, geometry=geometry)
sa = nk.sampler.MetropolisGaussian(hilb, sigma=0.04, n_chains=12, sweep_size=4)

mesh = Mesh(
    np.asarray(jax.devices()),
    axis_names=("S",),
    axis_types=(AxisType.Auto,),
)
jax.sharding.set_mesh(mesh)

potential = lambda x: ewald_coulomb(x, L, alpha=1.0, kmax=20)
epot = 1 / rs * nk.operator.PotentialEnergy(hilb, potential)
ekin = nk.operator.KineticEnergy(hilb, mass=rs**2)
ha = ekin + epot

model = MPNQS(n_per_spin=n_per_spin, sdim=3, L=L, rngs=nnx.Rngs(0))
vs = nk.vqs.MCState(sa, model, n_samples=8, n_discard_per_chain=2)
# vs.chunk_size = 1

op = nk.optimizer.Sgd(0.05)
sr = nk.optimizer.SR(diag_shift=0.0001)

log = nk.logging.JsonLog("HEG_14particles_test", save_params_every=7)

# gs = nk.VMC(ha, op, variational_state=vs, preconditioner=sr)
gs = nk.driver.VMC_SR(hamiltonian=ha, optimizer=op, variational_state=vs, diag_shift=0.01)
gs.run(n_iter=2, callback=mycb, out=log)


Automatic SR implementation choice: NTK (n_parameters=21603, n_samples=12)
Automatic SR on_the_fly choice: True
online_statistics: chain_length=1, exponential moving average window: 50, decay=0.980


  0%|          | 0/2 [00:00<?, ?it/s]

(JsonLog('HEG_14particles_test', mode=write, autoflush_cost=0.005)
   Runtime cost:
   	Log:    0.0016100406646728516
   	Params: 0.0025386810302734375,)

In [101]:
import netket
print(netket.__version__)

3.22.5.dev19+g5e4511b78
